In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# ---- metric.csv summary ----
metric_total_rows = df_metric.shape[0]
metric_cmdb_unique = pd.Series(df_metric['cmdb_id'].unique()[:50], name='cmdb_id')  # up to 50
metric_kpi_counts = (
    df_metric['kpi_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'kpi_name', 'kpi_name': 'count'})
    .head(50)  # limit to 50 distinct KPI names
)

# Compact metric summary
metric_summary = pd.DataFrame({
    'total_rows': [metric_total_rows],
    'distinct_cmdb_count': [df_metric['cmdb_id'].nunique()],
    'distinct_kpi_count': [df_metric['kpi_name'].nunique()]
})

# ---- trace.csv summary ----
trace_total_rows = df_trace.shape[0]
trace_cmdb_unique = pd.Series(df_trace['cmdb_id'].unique()[:50], name='cmdb_id')  # up to 50
trace_name_counts = (
    df_trace['trace_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
    .head(50)
)

trace_summary = pd.DataFrame({
    'total_rows': [trace_total_rows],
    'distinct_cmdb_count': [df_trace['cmdb_id'].nunique()],
    'distinct_trace_name_count': [df_trace['trace_name'].nunique()]
})

# ---- log.csv summary ----
log_total_rows = df_log.shape[0]
log_cmdb_unique = pd.Series(df_log['cmdb_id'].unique()[:50], name='cmdb_id')  # up to 50
log_name_counts = (
    df_log['log_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'log_name', 'log_name': 'count'})
    .head(50)
)

log_summary = pd.DataFrame({
    'total_rows': [log_total_rows],
    'distinct_cmdb_count': [df_log['cmdb_id'].nunique()],
    'distinct_log_name_count': [df_log['log_name'].nunique()]
})

# ---- error_logs.csv summary ----
err_distinct_cmdb = pd.Series(df_err['cmdb_id'].unique()[:50], name='cmdb_id')  # up to 50
err_top_messages = (
    df_err['message']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'message', 'message': 'count'})
    .head(20)  # top 20 messages by frequency
)

err_summary = pd.DataFrame({
    'total_rows': [df_err.shape[0]],
    'distinct_cmdb_count': [df_err['cmdb_id'].nunique()],
    'distinct_message_count': [df_err['message'].nunique()]
})

# Display compact summaries (variables separated by commas as required)
metric_summary, metric_cmdb_unique, metric_kpi_counts, trace_summary, trace_cmdb_unique, trace_name_counts, log_summary, log_cmdb_unique, log_name_counts, err_summary, err_distinct_cmdb, err_top_messages

```
Out[1]:
```
Summary of the loaded telemetry files:

- metric.csv
  - Total rows: 1,925
  - Distinct cmdb_id (15): carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db
  - Distinct KPI names (8) with counts:
    - cpu: 375
    - mem: 375
    - socket: 375
    - latency-50: 175
    - workload: 175
    - latency-90: 175
    - diskio: 175
    - error: 100

- trace.csv
  - Total rows: 0
  - No cmdb_id values present
  - No trace feature names present

- log.csv
  - Total rows: 458
  - Distinct cmdb_id (12): carts-db, catalogue, front-end, orders, payment, queue-master, shipping, user, carts, session-db, user-db, orders-db
  - Distinct log names (2) with counts:
    - log.error_count: 229
    - log.row_count: 229

- error_logs.csv
  - Total rows: 0
  - No cmdb_id values present
  - No log messages present

If you want, I can next: filter any of these by a time window, inspect specific KPIs or services, or search for particular error messages.

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  distinct_cmdb_count  distinct_kpi_count
0        1925                   15                   8, 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object,         count  count
0         cpu    375
1         mem    375
2      socket    375
3  latency-50    175
4    workload    175
5  latency-90    175
6      diskio    175
7       error    100,    total_rows  distinct_cmdb_count  distinct_trace_name_count
0           0                    0                          0, Series([], Name: cmdb_id, dtype: object), Empty DataFrame
Columns: [count, count]
Index: [],    total_rows  distinct_cmdb_count  distinct_log_name_count
0         458                   12                        2, 0         carts-db
1        catalogue
2        front-end
3           orders
4          payment
5     queue-master
6         shipping
7             user
8            carts
9       session-db
10         user-db
11       orders-db
Name: cmdb_id, dtype: object,              count  count
0  log.error_count    229
1    log.row_count    229,    total_rows  distinct_cmdb_count  distinct_message_count
0           0                    0                       0, Series([], Name: cmdb_id, dtype: object), Empty DataFrame
Columns: [count, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing df_metric if available, otherwise load and parse timestamp
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed (idempotent)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Group by (cmdb_id, kpi_name) and compute aggregated statistics over the full series
grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']

agg_stats = grp.agg(count='count', mean='mean', min='min', max='max')

# Compute percentiles
p95 = grp.quantile(0.95).rename('p95')
p90 = grp.quantile(0.90).rename('p90')
p5  = grp.quantile(0.05).rename('p5')

# Combine into a single DataFrame
agg_metric = pd.concat([agg_stats, p95, p90, p5], axis=1).reset_index()

# Sort by p95 descending and limit to top 50 rows
agg_metric_top50 = agg_metric.sort_values(by='p95', ascending=False).head(50)

# Total number of unique (cmdb_id, kpi_name) pairs
total_unique_pairs = agg_metric.shape[0]

# Keep output compact: show selected columns only
agg_metric_top50 = agg_metric_top50[['cmdb_id', 'kpi_name', 'count', 'mean', 'min', 'max', 'p95', 'p90', 'p5']]

# Display results (variable names as required)
agg_metric_top50, total_unique_pairs

```
Out[2]:
```
```python
# Summarize the previously computed aggregation (plain English string)
try:
    agg_metric_top50
except NameError:
    # fallback: recompute minimal if variable missing (shouldn't be needed)
    import pandas as pd
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    agg_stats = grp.agg(count='count', mean='mean', min='min', max='max')
    p95 = grp.quantile(0.95).rename('p95'); p90 = grp.quantile(0.90).rename('p90'); p5 = grp.quantile(0.05).rename('p5')
    agg_metric = pd.concat([agg_stats, p95, p90, p5], axis=1).reset_index()
    agg_metric_top50 = agg_metric.sort_values(by='p95', ascending=False).head(50)
    total_unique_pairs = agg_metric.shape[0]

top5 = agg_metric_top50.head(5).copy()

lines = []
lines.append(f"Total unique (cmdb_id, kpi_name) pairs: {total_unique_pairs}.")
lines.append("Top 5 (cmdb_id, kpi_name) by p95 (with count, mean, p95, p90):")

for i, row in enumerate(top5.itertuples(index=False), start=1):
    cmdb = row.cmdb_id
    kpi = row.kpi_name
    cnt = int(row.count)
    mean = f"{row.mean:.6g}"
    p95v = f"{row.p95:.6g}"
    p90v = f"{row.p90:.6g}"
    lines.append(f"{i}. {cmdb} / {kpi} — count={cnt}, mean={mean}, p95={p95v}, p90={p90v}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  count          mean           min           max           p95           p90            p5
20       catalogue-db      mem     25  4.324571e+08  4.324516e+08  4.324917e+08  4.324816e+08  4.324671e+08  4.324517e+08
33             orders      mem     25  3.300709e+08  3.298932e+08  3.307601e+08  3.305185e+08  3.303835e+08  3.298966e+08
64           shipping      mem     25  3.066794e+08  3.064854e+08  3.076014e+08  3.073605e+08  3.066431e+08  3.064877e+08
48       queue-master      mem     25  2.674920e+08  2.673912e+08  2.678311e+08  2.678276e+08  2.678263e+08  2.673935e+08
4               carts      mem     25  2.164544e+08  2.164277e+08  2.165219e+08  2.165133e+08  2.165006e+08  2.164295e+08
..                ...      ...    ...           ...           ...           ...           ...           ...           ...
0               carts      cpu     25  1.381292e+00  1.278158e+00  1.501177e+00  1.452604e+00  1.442882e+00  1.290987e+00
73            user-db      cpu     25  1.055735e+00  9.221444e-01  1.164579e+00  1.155703e+00  1.149682e+00  9.378727e-01
61           shipping      cpu     25  3.722782e-01  2.132910e-01  1.703709e+00  1.145648e+00  4.742500e-01  2.278676e-01
67               user      cpu     25  8.899422e-01  7.605589e-01  1.063772e+00  1.045521e+00  1.042641e+00  7.749103e-01
56  rabbitmq-exporter   socket     25  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00

[50 rows x 9 columns], 77)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_metric if available; otherwise load and parse timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Reuse agg_metric if available; otherwise compute global thresholds (must be computed on full series BEFORE filtering)
try:
    agg_metric
except NameError:
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    agg_stats = grp.agg(count='count', mean='mean', min='min', max='max')
    p95 = grp.quantile(0.95).rename('p95')
    p90 = grp.quantile(0.90).rename('p90')
    p5  = grp.quantile(0.05).rename('p5')
    agg_metric = pd.concat([agg_stats, p95, p90, p5], axis=1).reset_index()

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-18 19:28:30", utc=True)
end_ts   = pd.to_datetime("2024-01-18 19:58:30", utc=True)

# Filter metric rows to the incident window inclusive
df_win = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge global thresholds into windowed data
# agg_metric contains p95 and p5 per (cmdb_id, kpi_name)
df_win = df_win.merge(agg_metric[['cmdb_id','kpi_name','p95','p5']], on=['cmdb_id','kpi_name'], how='left')

# Determine anomaly per row:
# - workload: anomaly if value <= p5 (low-is-bad)
# - others: anomaly if value >= p95 (high-is-bad)
def detect_anomaly(row):
    p95 = row['p95']
    p5 = row['p5']
    v = row['value']
    if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
        return False
    if row['kpi_name'] == 'workload':
        # require p5 not NaN
        return False if pd.isna(p5) else (v <= p5)
    else:
        return False if pd.isna(p95) else (v >= p95)

df_win['anomaly'] = df_win.apply(detect_anomaly, axis=1)

# Aggregate per (cmdb_id, kpi_name)
aggregations = {
    'anomaly': 'sum',
    'timestamp': [lambda x: x.min(), lambda x: x.max()],
    'value': ['max', 'min'],
    'p95': 'first',
    'p5': 'first'
}
agg = df_win.groupby(['cmdb_id','kpi_name']).agg(aggregations)

# Flatten MultiIndex columns
agg.columns = ['anomaly_count', 'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
               'max_value_in_window', 'min_value_in_window', 'p95', 'p5']
agg = agg.reset_index()

# Convert earliest/latest timestamps from group to NaT if anomaly_count == 0
# (they may be timestamps of non-anomalous rows if grouping included; but we used timestamps of all rows,
# so ensure we pick timestamps only among anomalous rows)
# Recompute earliest/latest strictly from anomalous rows to be safe
anom_rows = df_win[df_win['anomaly']]
if not anom_rows.empty:
    times = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].agg(['min','max']).reset_index()
    times = times.rename(columns={'min':'earliest_anomaly_timestamp','max':'latest_anomaly_timestamp'})
    agg = agg.drop(columns=['earliest_anomaly_timestamp','latest_anomaly_timestamp']).merge(times, on=['cmdb_id','kpi_name'], how='left')
else:
    # no anomalies at all: set columns to NaT
    agg['earliest_anomaly_timestamp'] = pd.NaT
    agg['latest_anomaly_timestamp'] = pd.NaT

# Ensure numeric columns typed properly
for col in ['anomaly_count','max_value_in_window','min_value_in_window','p95','p5']:
    if col in agg.columns:
        agg[col] = pd.to_numeric(agg[col], errors='coerce')

# Compute severity_metric:
# - for workload (low-is-bad): severity = p5 / min_value_in_window
# - otherwise: severity = max_value_in_window / p95
def compute_severity(row):
    if row['anomaly_count'] <= 0:
        return np.nan
    if row['kpi_name'] == 'workload':
        denom = row['min_value_in_window']
        num = row['p5']
    else:
        denom = row['p95']
        num = row['max_value_in_window']
    # safe division
    if pd.isna(num) or pd.isna(denom):
        return np.nan
    if denom == 0:
        return np.inf if num > 0 else np.nan
    return float(num) / float(denom)

agg['severity_metric'] = agg.apply(compute_severity, axis=1)

# Keep only pairs that had anomaly_count > 0
agg_anom = agg[agg['anomaly_count'] > 0].copy()

# Select and order columns as requested
result_cols = ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','latest_anomaly_timestamp',
               'max_value_in_window','min_value_in_window','p95','p5','severity_metric']
agg_anom = agg_anom[result_cols]

# Sort by anomaly_count desc then severity_metric desc, limit to top 20
result_top20 = agg_anom.sort_values(by=['anomaly_count','severity_metric'], ascending=[False, False]).head(20)

# Total number of (cmdb_id,kpi_name) pairs with anomaly_count > 0
anomaly_pairs_count = agg_anom.shape[0]

# Return compact outputs
result_top20, anomaly_pairs_count

```
Out[3]:
```
Summary of anomalies in the incident window (2024-01-18 19:28:30–19:58:30 UTC):

- Total number of (cmdb_id, kpi_name) pairs that showed at least one anomaly: 77.

- Highest-volume anomalies (anomaly_count = 25 each, occurring roughly between 2024-01-18 19:31:00 and 19:55:00 UTC):
  - Many components reported socket breaches: carts-db (socket), catalogue (socket), catalogue-db (socket), queue-master (socket), rabbitmq (socket), rabbitmq-exporter (socket), session-db (socket), user (socket), user-db (socket), orders-db (socket), payment (socket).
  - rabbitmq-exporter also had mem anomalies (mem).
  - Several components reported error or diskio values equal to their historical p95/p5 thresholds (error and diskio rows show anomaly_count=25 but severity undefined because thresholds are zero).

- Highest severity anomalies (severity_metric descending, notable examples):
  - shipping / latency-90 — anomaly_count=2, severity ≈ 18.34 (max in window ≈ 0.08665 vs p95 ≈ 0.004715). This is a large breach of the p95 threshold.
  - orders / error — anomaly_count=2, severity ≈ 1.68.
  - shipping / cpu — anomaly_count=2, severity ≈ 1.49.
  - payment / latency-90 — anomaly_count=2, severity ≈ 1.46.

- Observations about severity values:
  - Many socket anomalies have severity = 1.0 (max_value == p95), i.e., the observed peak equals the global p95 threshold.
  - Some metrics (error, diskio) have p95 == 0, so severity is undefined (NaN) even though anomaly_count is high — these indicate values at or above a zero threshold and require special handling/interpretation.
  - The most striking outlier by ratio is shipping / latency-90 (very large multiple of the p95).

- Timing:
  - Earliest anomalous timestamps for the major groups are around 2024-01-18 19:31:00 UTC.
  - Latest anomalies for many persistent cases run through ~2024-01-18 19:55:00 UTC; some components had shorter windows (e.g., user/user-db anomalies end ~19:43:00).

Bottom line:
- The event shows widespread socket-related breaches across many services (high count, many sustained during the window) plus a small set of high-severity outliers (notably shipping latency-90). Investigate socket/network conditions across the affected services and dig into shipping traces/logs for the latency outlier.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window           p95            p5  severity_metric
10           carts-db      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         8.000000e+00         8.000000e+00  8.000000e+00  8.000000e+00         1.000000
16          catalogue      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         7.000000e+00         7.000000e+00  7.000000e+00  7.000000e+00         1.000000
21       catalogue-db      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00         1.000000
49       queue-master      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00         1.000000
53           rabbitmq      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         1.100000e+01         1.100000e+01  1.100000e+01  1.100000e+01         1.000000
55  rabbitmq-exporter         mem             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         4.575232e+06         4.575232e+06  4.575232e+06  4.575232e+06         1.000000
56  rabbitmq-exporter      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00         1.000000
60         session-db      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00         1.000000
1               carts       error             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00              NaN
12          catalogue       error             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00              NaN
19       catalogue-db      diskio             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00              NaN
51           rabbitmq      diskio             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00              NaN
71               user      socket             13  2024-01-18 19:31:00+00:00 2024-01-18 19:43:00+00:00         1.900000e+01         1.800000e+01  1.900000e+01  1.800000e+01         1.000000
76            user-db      socket             13  2024-01-18 19:31:00+00:00 2024-01-18 19:43:00+00:00         9.000000e+00         8.000000e+00  9.000000e+00  8.000000e+00         1.000000
39          orders-db      socket              7  2024-01-18 19:49:00+00:00 2024-01-18 19:55:00+00:00         6.000000e+00         5.000000e+00  6.000000e+00  5.000000e+00         1.000000
44            payment      socket              7  2024-01-18 19:49:00+00:00 2024-01-18 19:55:00+00:00         5.000000e+00         4.000000e+00  5.000000e+00  4.000000e+00         1.000000
63           shipping  latency-90              2  2024-01-18 19:36:00+00:00 2024-01-18 19:46:00+00:00         8.646667e-02         4.600000e-03  4.715084e-03  4.600000e-03        18.338309
30             orders       error              2  2024-01-18 19:46:00+00:00 2024-01-18 19:55:00+00:00         4.578667e-01         0.000000e+00  2.725632e-01  0.000000e+00         1.679855
61           shipping         cpu              2  2024-01-18 19:36:00+00:00 2024-01-18 19:54:00+00:00         1.703709e+00         2.132910e-01  1.145648e+00  2.278676e-01         1.487114
42            payment  latency-90              2  2024-01-18 19:41:00+00:00 2024-01-18 19:46:00+00:00         6.449024e-03         3.596928e-03  4.412934e-03  3.660927e-03         1.461391, 77)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse df_log if available, otherwise load and parse timestamp
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-18 19:28:30", utc=True)
end_ts   = pd.to_datetime("2024-01-18 19:58:30", utc=True)

# Filter to window inclusive
df_win = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# Aggregate sums for the two log_name types per cmdb_id
sum_error = df_win[df_win['log_name'] == 'log.error_count'].groupby('cmdb_id')['value'].sum().rename('sum_log_error_count')
sum_row   = df_win[df_win['log_name'] == 'log.row_count'].groupby('cmdb_id')['value'].sum().rename('sum_log_row_count')

# Rows in window per cmdb_id
rows_in_window = df_win.groupby('cmdb_id').size().rename('rows_in_window')

# Earliest and latest timestamps per cmdb_id
times = df_win.groupby('cmdb_id')['timestamp'].agg(['min','max']).rename(columns={'min':'earliest_timestamp','max':'latest_timestamp'})

# Combine into one DataFrame
summary = pd.concat([sum_error, sum_row, rows_in_window, times], axis=1).fillna(0)

# Ensure numeric sums are proper dtype
summary['sum_log_error_count'] = pd.to_numeric(summary['sum_log_error_count'], errors='coerce').fillna(0)
summary['sum_log_row_count'] = pd.to_numeric(summary['sum_log_row_count'], errors='coerce').fillna(0)
summary['rows_in_window'] = pd.to_numeric(summary['rows_in_window'], errors='coerce').fillna(0).astype(int)

# Reset index to have cmdb_id as a column
summary = summary.reset_index()

# Sort as requested and limit to 20 rows
result = summary.sort_values(by=['sum_log_error_count','rows_in_window'], ascending=[False, False]).head(20)

# cmdb_id values where sum_log_error_count > 0 (as a Series, bounded)
cmdbs_with_errors = result[result['sum_log_error_count'] > 0]['cmdb_id']

# Return compact results
result, cmdbs_with_errors

```
Out[4]:
```
Summary of log activity in the incident window (2024-01-18 19:28:30 → 19:58:30 UTC):

- No services reported any log.error_count > 0 in the window (empty list).

- Per-cmdb highlights (limited to 20 rows shown; times are UTC):
  - front-end — sum_log_row_count: 46,570; rows_in_window: 50; earliest 2024-01-18 19:31:00; latest 2024-01-18 19:55:00
  - user — sum_log_row_count: 14,029; rows_in_window: 50; earliest 19:31:00; latest 19:55:00
  - queue-master — sum_log_row_count: 12,510; rows_in_window: 50; earliest 19:31:00; latest 19:55:00
  - orders — sum_log_row_count: 6,788; rows_in_window: 50; earliest 19:31:00; latest 19:55:00
  - catalogue — sum_log_row_count: 3,866; rows_in_window: 50; earliest 19:31:00; latest 19:55:00
  - payment — sum_log_row_count: 2,367; rows_in_window: 50; earliest 19:31:00; latest 19:55:00
  - carts — sum_log_row_count: 1,650; rows_in_window: 46; earliest 19:32:00; latest 19:55:00
  - shipping — sum_log_row_count: 1,390; rows_in_window: 50; earliest 19:31:00; latest 19:55:00
  - carts-db — sum_log_row_count: 27; rows_in_window: 48; earliest 19:31:00; latest 19:54:00
  - session-db — sum_log_row_count: 25; rows_in_window: 10; earliest 19:35:00; latest 19:55:00
  - orders-db — sum_log_row_count: 7; rows_in_window: 2; earliest/latest 19:48:00
  - user-db — sum_log_row_count: 1; rows_in_window: 2; earliest/latest 19:44:00

- Bottom line: high log volume in front-end, user, queue-master, orders and catalogue, but no recorded log.error_count in this incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  sum_log_error_count  sum_log_row_count  rows_in_window        earliest_timestamp          latest_timestamp
2      catalogue                    0               3866              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
3      front-end                    0              46570              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
4         orders                    0               6788              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
6        payment                    0               2367              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
7   queue-master                    0              12510              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
9       shipping                    0               1390              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
10          user                    0              14029              50 2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00
1       carts-db                    0                 27              48 2024-01-18 19:31:00+00:00 2024-01-18 19:54:00+00:00
0          carts                    0               1650              46 2024-01-18 19:32:00+00:00 2024-01-18 19:55:00+00:00
8     session-db                    0                 25              10 2024-01-18 19:35:00+00:00 2024-01-18 19:55:00+00:00
5      orders-db                    0                  7               2 2024-01-18 19:48:00+00:00 2024-01-18 19:48:00+00:00
11       user-db                    0                  1               2 2024-01-18 19:44:00+00:00 2024-01-18 19:44:00+00:00, Series([], Name: cmdb_id, dtype: object))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_metric and agg_metric if available; otherwise load/compute them
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    agg_metric
except NameError:
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    agg_stats = grp.agg(count='count', mean='mean', min='min', max='max')
    p95 = grp.quantile(0.95).rename('p95')
    p5  = grp.quantile(0.05).rename('p5')
    agg_metric = pd.concat([agg_stats, p95, p5], axis=1).reset_index()

# Incident window and candidate components
start_ts = pd.to_datetime("2024-01-18 19:28:30", utc=True)
end_ts   = pd.to_datetime("2024-01-18 19:58:30", utc=True)
candidates = ['carts', 'catalogue', 'orders', 'payment', 'user']

# Filter metric rows to window and candidate cmdb_ids
df_cand_win = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(candidates))
].copy()

# Merge global thresholds (p95, p5) computed on full series (not filtered)
df_cand_win = df_cand_win.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Classify anomalies:
# workload: anomaly if value <= p5 (low-is-bad)
# others: anomaly if value >= p95 (high-is-bad)
cond_workload = (df_cand_win['kpi_name'] == 'workload') & df_cand_win['value'].le(df_cand_win['p5'])
cond_highbad = (df_cand_win['kpi_name'] != 'workload') & df_cand_win['value'].ge(df_cand_win['p95'])
df_cand_win['anomaly'] = (cond_workload | cond_highbad).fillna(False)

# Aggregations:
# - max/min in window per (cmdb_id,kpi_name) computed from window rows
win_maxmin = df_cand_win.groupby(['cmdb_id','kpi_name'])['value'].agg(max_value_in_window='max', min_value_in_window='min').reset_index()

# - anomaly counts and earliest/latest anomaly timestamps (only from anomalous rows)
anom_rows = df_cand_win[df_cand_win['anomaly']]
if not anom_rows.empty:
    anom_agg = anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count=('anomaly','sum'),
        earliest_anomaly_timestamp=('timestamp','min'),
        latest_anomaly_timestamp=('timestamp','max')
    ).reset_index()
else:
    # empty frame
    anom_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','latest_anomaly_timestamp'])

# Combine with thresholds and max/min
combined = pd.merge(win_maxmin, agg_metric[['cmdb_id','kpi_name','p95','p5']], on=['cmdb_id','kpi_name'], how='left')
combined = pd.merge(combined, anom_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill anomaly_count NaN with 0 and keep timestamp NaT for no anomalies
combined['anomaly_count'] = combined['anomaly_count'].fillna(0).astype(int)

# Compute severity_metric
def severity(row):
    if row['anomaly_count'] <= 0:
        return np.nan
    kpi = row['kpi_name']
    if kpi == 'workload':
        # severity = p5 / min_value_in_window
        num = row['p5']
        denom = row['min_value_in_window']
    else:
        # severity = max_value_in_window / p95
        num = row['max_value_in_window']
        denom = row['p95']
    if pd.isna(num) or pd.isna(denom):
        return np.nan
    if denom == 0:
        return np.inf if num > 0 else np.nan
    return float(num) / float(denom)

combined['severity_metric'] = combined.apply(severity, axis=1)

# Select requested columns and sort
result = combined[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','latest_anomaly_timestamp',
                   'max_value_in_window','min_value_in_window','p95','p5','severity_metric']]

result_top20 = result.sort_values(by=['anomaly_count','severity_metric'], ascending=[False, False]).head(20)

# Determine which candidate component has the earliest anomaly timestamp among all their KPIs
# Compute per-component earliest anomaly timestamp (ignore components with no anomalies)
comp_earliest = result[result['anomaly_count']>0].groupby('cmdb_id', as_index=False)['earliest_anomaly_timestamp'].min()
if not comp_earliest.empty:
    comp_earliest = comp_earliest.sort_values('earliest_anomaly_timestamp').reset_index(drop=True)
    earliest_component_info = comp_earliest.iloc[[0]]  # single-row DataFrame with cmdb_id and timestamp
else:
    earliest_component_info = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_timestamp'])

# Return compact outputs (DataFrame and earliest component info)
result_top20, earliest_component_info

```
Out[5]:
```
summary = (
    "Summary for candidates (carts, catalogue, orders, payment, user) during 2024-01-18 19:28:30–19:58:30 UTC:\n\n"
    "- Sustained, high-count anomalies:\n"
    "  • catalogue / socket — anomaly_count=25 (19:31–19:55 UTC), p95=7, severity=1.0\n"
    "  • carts / error — anomaly_count=25 (19:31–19:55 UTC), p95=0 (severity undefined)\n"
    "  • catalogue / error — anomaly_count=25 (19:31–19:55 UTC), p95=0 (severity undefined)\n"
    "  • user / socket — anomaly_count=13 (19:31–19:43 UTC), p95=19, severity=1.0\n"
    "  • payment / socket — anomaly_count=7 (19:49–19:55 UTC), p95=5, severity=1.0\n\n"
    "- Notable lower-volume but high-severity breaches (by ratio):\n"
    "  • orders / error — anomaly_count=2, severity≈1.68 (max≈0.4579 vs p95≈0.2726)\n"
    "  • payment / latency-90 — anomaly_count=2, severity≈1.46\n"
    "  • orders / cpu — anomaly_count=2, severity≈1.38\n\n"
    "- Many socket anomalies have severity = 1.0 (observed peaks equal the global p95).\n"
    "- Some metrics (error) have p95 == 0, producing undefined severity despite high anomaly counts; these require special handling.\n\n"
    "- Earliest anomaly among these candidate components: carts at 2024-01-18 19:31:00 UTC.\n\n"
    "Recommendation: prioritize investigating networking/socket conditions (widespread socket breaches across services) and examine logs/traces for catalogue and carts; also inspect orders/payment for the moderate-severity CPU/latency/error spikes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window        p95         p5  severity_metric
12  catalogue      socket             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00             7.000000             7.000000   7.000000   7.000000         1.000000
1       carts       error             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00             0.000000             0.000000   0.000000   0.000000              NaN
8   catalogue       error             25  2024-01-18 19:31:00+00:00 2024-01-18 19:55:00+00:00             0.000000             0.000000   0.000000   0.000000              NaN
31       user      socket             13  2024-01-18 19:31:00+00:00 2024-01-18 19:43:00+00:00            19.000000            18.000000  19.000000  18.000000         1.000000
25    payment      socket              7  2024-01-18 19:49:00+00:00 2024-01-18 19:55:00+00:00             5.000000             4.000000   5.000000   4.000000         1.000000
15     orders       error              2  2024-01-18 19:46:00+00:00 2024-01-18 19:55:00+00:00             0.457867             0.000000   0.272563   0.000000         1.679855
23    payment  latency-90              2  2024-01-18 19:41:00+00:00 2024-01-18 19:46:00+00:00             0.006449             0.003597   0.004413   0.003661         1.461391
14     orders         cpu              2  2024-01-18 19:46:00+00:00 2024-01-18 19:53:00+00:00             2.609279             0.875315   1.885867   0.923563         1.383596
16     orders  latency-50              2  2024-01-18 19:50:00+00:00 2024-01-18 19:54:00+00:00             0.814968             0.037473   0.741779   0.037504         1.098667
7   catalogue         cpu              2  2024-01-18 19:32:00+00:00 2024-01-18 19:55:00+00:00             0.194691             0.153179   0.177626   0.160104         1.096071
26    payment    workload              2  2024-01-18 19:48:00+00:00 2024-01-18 19:55:00+00:00             2.132133             0.800065   2.111283   0.853213         1.066431
5       carts      socket              2  2024-01-18 19:51:00+00:00 2024-01-18 19:55:00+00:00            13.903226            12.000000  13.223333  12.036667         1.051416
28       user  latency-50              2  2024-01-18 19:46:00+00:00 2024-01-18 19:51:00+00:00             0.004520             0.003190   0.004314   0.003213         1.047773
29       user  latency-90              2  2024-01-18 19:46:00+00:00 2024-01-18 19:51:00+00:00             0.009117             0.004943   0.008788   0.005033         1.037443
0       carts         cpu              2  2024-01-18 19:31:00+00:00 2024-01-18 19:46:00+00:00             1.501177             1.278158   1.452604   1.290987         1.033439
17     orders  latency-90              2  2024-01-18 19:50:00+00:00 2024-01-18 19:55:00+00:00             3.737903             0.047502   3.637303   0.047523         1.027658
22    payment  latency-50              2  2024-01-18 19:41:00+00:00 2024-01-18 19:42:00+00:00             0.002073             0.000356   0.002019   0.000358         1.027016
6       carts    workload              2  2024-01-18 19:47:00+00:00 2024-01-18 19:50:00+00:00             8.476633             6.533117   8.389433   6.688153         1.023731
27       user         cpu              2  2024-01-18 19:46:00+00:00 2024-01-18 19:47:00+00:00             1.063772             0.760559   1.045521   0.774910         1.017456
19     orders      socket              2  2024-01-18 19:45:00+00:00 2024-01-18 19:48:00+00:00            22.650000            17.000000  22.320000  17.960000         1.014785,   cmdb_id earliest_anomaly_timestamp
0   carts  2024-01-18 19:31:00+00:00)```
```